In [10]:
import mne 
import matplotlib.pyplot as plt
import pandas as pd

subID = "001"
sesID = "001"
device = "portalite"
modality = "concentric"

mne.viz.set_browser_backend("qt")

raw = mne.io.read_raw_snirf(f"""\\Program Files\\DigiMove\\DigiMove\\DataAnalysisProject\\data\\MOXY-bids\\sub-{subID}\\ses-{sesID}\\nirs\\sub-{subID}_ses-{sesID}_task-{modality}_recording-{device}_nirs.snirf""", preload = True)
#set DPF
duration = raw.times[-1]

events_filepath = f"""\\Program Files\\DigiMove\\DigiMove\\DataAnalysisProject\\data\\MOXY-bids\\sub-{subID}\\ses-{sesID}\\beh\\sub-{subID}_ses-{sesID}_task-{modality}_events.tsv"""
events = pd.read_csv(events_filepath, sep="\t")
print(events)

markers = mne.Annotations(
    onset=events["onset"].values,
    duration=events["duration"].values,
    description=events["trial_type"].values
    )
raw.set_annotations(markers)  

#Définir des segments, pour pouvoir les afficher uniquement


# 1. Convertir intensité → optical density
raw_od = mne.preprocessing.nirs.optical_density(raw)


# 2. Convertir OD → HbO / HbR via Beer-Lambert
raw_hb = mne.preprocessing.nirs.beer_lambert_law(raw_od, ppf=0.25)
print(raw_hb)
#raw_hb.plot(duration=duration)

# 3. Filtrer les données with a low-pass filter (cutoff at 5 Hz) and a butterworth filter of order 4.
raw_hb_filt = raw_hb.copy().filter(
    l_freq=0.01,
    h_freq=4,
    method="iir",
    iir_params=dict(order=4, ftype="butter"),
)
raw_hb_filt.plot(duration=duration)

# 4. Display 20 secondes after the first event conc50. 
event_time = events[events["trial_type"] == "Con50"]["onset"].values[0]

#5. center around 0 and take the air under the curve
#df['rab_hb_filt']


raw_hb_filt.plot(start=event_time, duration=20)





Loading c:\Program Files\DigiMove\DigiMove\DataAnalysisProject\data\MOXY-bids\sub-001\ses-001\nirs\sub-001_ses-001_task-concentric_recording-portalite_nirs.snirf
Found jitter of 0.000000% in sample times.
Reading 0 ... 22351  =      0.000 ...  2235.100 secs...
         onset  duration trial_type
0   918.723293         0        MVC
1   992.726686         0        MVC
2  1066.132068         0        MVC
3  1260.835733         0      Con50
4  1389.520155         0      Con50
5  1519.444601         0      Con50
6  1732.394609         0      Con30
7  1862.235053         0      Con30
8  1991.915494         0      Con30
9  2206.353531         0        MVC
<RawSNIRF | sub-001_ses-001_task-concentric_recording-portalite_nirs.snirf, 6 x 22352 (2235.2 s), ~1.0 MiB, data loaded>
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.01 - 4 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filte